In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from dataloading import get_data_loaders 
import torchvision.models as models


In [22]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [23]:
batch_size = 16
max_samples = 500  
train_loader, val_loader = get_data_loaders(batch_size=batch_size, max_samples=max_samples)

In [24]:
num_classes = 1
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

In [25]:
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [26]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

In [27]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device).float()
        labels = labels.unsqueeze(1)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loop.set_postfix(loss=loss.item())

Epoch 5: 100%|██████████| 32/32 [00:27<00:00,  1.16it/s, loss=0.0267]


In [28]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy}%")

Validation Accuracy: 100.0%


In [29]:
torch.save(model.state_dict(), "resnet18_best.pth")